In [7]:
# Cell 1: LLM generates Cypher using prompt_1 + schema + user question
import os
from neo4j import GraphDatabase
from openai import OpenAI

PROMPT_1_PATH = '/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/agentic_cyph_retr_cycle_sandbox/prompt_1_q1.txt'
# QUESTION = 'Which documents do I need to bring to passport control in France?'
QUESTION = 'Which countries are mentioned in database?'



NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7688')
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = '12345678'

DEEPSEEK_API_KEY = 'sk-8da8031877dc4c39ae8d7b624c70f01f'
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL', 'deepseek-reasoner')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com/v1')

def fetch_schema_text():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run('CALL db.schema.visualization()')
        record = result.single()
    driver.close()

    nodes = record['nodes']
    rels = record['relationships']

    labels = sorted({lbl for n in nodes for lbl in n.labels})
    lines = ['NODE LABELS:']
    lines += [f'- {l}' for l in labels]
    lines.append('')
    lines.append('RELATIONSHIPS:')
    for r in rels:
        s = ':'.join(r.start_node.labels) or ''
        e = ':'.join(r.end_node.labels) or ''
        lines.append(f'- (:{s})-[:{r.type}]->(:{e})')
    return ''.join(lines)

def build_prompt(prompt_template, schema_text, question):
    prompt = prompt_template.replace('{SCHEMA_TEXT}', schema_text).replace('{SCHEMA}', schema_text)
    prompt = prompt.replace('{USER_QUERY}', question).replace('{USER_QUESTION}', question)
    if '{SCHEMA_TEXT}' not in prompt_template and '{SCHEMA}' not in prompt_template:
        prompt += f"DATABASE SCHEMA{schema_text}"
    if '{USER_QUERY}' not in prompt_template and '{USER_QUESTION}' not in prompt_template:
        prompt += f"QUESTION {question}"
    return prompt

def call_llm(prompt):
    client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

with open(PROMPT_1_PATH, 'r', encoding='utf-8') as f:
    prompt_1_template = f.read()

schema_text = fetch_schema_text()
prompt_1 = build_prompt(prompt_1_template, schema_text, QUESTION)
cypher = call_llm(prompt_1)
print(cypher)

cypher = '{"cypher":"MATCH (c:Countries) RETURN DISTINCT c LIMIT 200"}'
print(cypher)


{"cypher":"MATCH (c:Country) RETURN DISTINCT c LIMIT 200"}
{"cypher":"MATCH (c:Countries) RETURN DISTINCT c LIMIT 200"}


In [10]:
# Cell 2: Execute Cypher
from neo4j import GraphDatabase

EXEC_ERROR = False

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    with driver.session() as session:
        data = session.run(cypher).data()
    print(data)
except Exception:
    EXEC_ERROR = True
finally:
    driver.close()
# Cell 3: If execution error, print ОШИБКА
if EXEC_ERROR:
    print('ОШИБКА')


ОШИБКА
